<a href="https://colab.research.google.com/github/kalsep/Data-Engineering-Interview-Preperations/blob/main/Windows_Function_in_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.functions import countDistinct, coalesce, lit, to_date

In [ ]:
spark = SparkSession.builder.appName('Basics').getOrCreate()

In [ ]:
# Create Sample Dataframe
employees = [
    (7369, "SMITH", "CLERK", "17-Dec-80", 800, 20, 10),
    (7499, "ALLEN", "SALESMAN", "20-Feb-81", 1600, 300, 30),
    (7521, "WARD", "SALESMAN", "22-Feb-81", 1250, 500, 30),
    (7566, "JONES", "MANAGER", "2-Apr-81", 2975, 0, 20),
    (7654, "MARTIN", "SALESMAN", "28-Sep-81", 1250, 1400, 30),
    (7698, "BLAKE", "MANAGER", "1-May-81", 2850, 0, 30),
    (7782, "CLARK", "MANAGER", "9-Jun-81", 2450, 0, 10),
    (7788, "SCOTT", "ANALYST", "19-Apr-87", 3000, 0, 20),
    (7629, "ALEX", "SALESMAN", "28-Sep-79", 1150, 1400, 30),
    (7839, "KING", "PRESIDENT", "17-Nov-81", 5000, 0, 10),
    (7844, "TURNER", "SALESMAN", "8-Sep-81", 1500, 0, 30),
    (7876, "ADAMS", "CLERK", "23-May-87", 1100, 0, 20)
]

In [ ]:
emp_df = spark.createDataFrame(employees, ['empno', 'ename', 'job', 'hiredate', 'sal', 'comm', 'deptno'])

In [ ]:
emp_df.show()

+-----+------+---------+---------+----+----+------+
|empno| ename|      job| hiredate| sal|comm|deptno|
+-----+------+---------+---------+----+----+------+
| 7369| SMITH|    CLERK|17-Dec-80| 800|  20|    10|
| 7499| ALLEN| SALESMAN|20-Feb-81|1600| 300|    30|
| 7521|  WARD| SALESMAN|22-Feb-81|1250| 500|    30|
| 7566| JONES|  MANAGER| 2-Apr-81|2975|   0|    20|
| 7654|MARTIN| SALESMAN|28-Sep-81|1250|1400|    30|
| 7698| BLAKE|  MANAGER| 1-May-81|2850|   0|    30|
| 7782| CLARK|  MANAGER| 9-Jun-81|2450|   0|    10|
| 7788| SCOTT|  ANALYST|19-Apr-87|3000|   0|    20|
| 7629|  ALEX| SALESMAN|28-Sep-79|1150|1400|    30|
| 7839|  KING|PRESIDENT|17-Nov-81|5000|   0|    10|
| 7844|TURNER| SALESMAN| 8-Sep-81|1500|   0|    30|
| 7876| ADAMS|    CLERK|23-May-87|1100|   0|    20|
+-----+------+---------+---------+----+----+------+



In [ ]:
emp_df.printSchema()

root
 |-- empno: long (nullable = true)
 |-- ename: string (nullable = true)
 |-- job: string (nullable = true)
 |-- hiredate: string (nullable = true)
 |-- sal: long (nullable = true)
 |-- comm: long (nullable = true)
 |-- deptno: long (nullable = true)



### Creating a View

# Rank()

In [ ]:
emp_df.createOrReplaceTempView("emp")

In [ ]:
ranked_df = spark.sql("""
SELECT empno,ename,job,deptno,sal, RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS rank
FROM emp
""")

In [ ]:
ranked_df.show()

+-----+------+---------+------+----+----+
|empno| ename|      job|deptno| sal|rank|
+-----+------+---------+------+----+----+
| 7839|  KING|PRESIDENT|    10|5000|   1|
| 7782| CLARK|  MANAGER|    10|2450|   2|
| 7369| SMITH|    CLERK|    10| 800|   3|
| 7788| SCOTT|  ANALYST|    20|3000|   1|
| 7566| JONES|  MANAGER|    20|2975|   2|
| 7876| ADAMS|    CLERK|    20|1100|   3|
| 7698| BLAKE|  MANAGER|    30|2850|   1|
| 7499| ALLEN| SALESMAN|    30|1600|   2|
| 7844|TURNER| SALESMAN|    30|1500|   3|
| 7521|  WARD| SALESMAN|    30|1250|   4|
| 7654|MARTIN| SALESMAN|    30|1250|   4|
| 7629|  ALEX| SALESMAN|    30|1150|   6|
+-----+------+---------+------+----+----+



#### Using PYspark

In [ ]:
from pyspark.sql.functions import col, rank
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [ ]:
windowSpec = Window.partitionBy(col('deptno')).orderBy(col('sal').desc())

In [ ]:
ranked_result_df = emp_df.select('empno', 'ename', 'job', 'deptno', 'sal',F.rank().over(windowSpec).alias('rank'))

In [ ]:
ranked_result_df.show()

+-----+------+---------+------+----+----+
|empno| ename|      job|deptno| sal|rank|
+-----+------+---------+------+----+----+
| 7839|  KING|PRESIDENT|    10|5000|   1|
| 7782| CLARK|  MANAGER|    10|2450|   2|
| 7369| SMITH|    CLERK|    10| 800|   3|
| 7788| SCOTT|  ANALYST|    20|3000|   1|
| 7566| JONES|  MANAGER|    20|2975|   2|
| 7876| ADAMS|    CLERK|    20|1100|   3|
| 7698| BLAKE|  MANAGER|    30|2850|   1|
| 7499| ALLEN| SALESMAN|    30|1600|   2|
| 7844|TURNER| SALESMAN|    30|1500|   3|
| 7521|  WARD| SALESMAN|    30|1250|   4|
| 7654|MARTIN| SALESMAN|    30|1250|   4|
| 7629|  ALEX| SALESMAN|    30|1150|   6|
+-----+------+---------+------+----+----+



# Dense rank

In [ ]:
dense_df = spark.sql(
        """SELECT empno, ename, job, deptno, sal,
        DENSE_RANK() OVER (PARTITION BY deptno ORDER BY sal DESC)
        AS dense_rank FROM emp""")
dense_df.show()

+-----+------+---------+------+----+----------+
|empno| ename|      job|deptno| sal|dense_rank|
+-----+------+---------+------+----+----------+
| 7839|  KING|PRESIDENT|    10|5000|         1|
| 7782| CLARK|  MANAGER|    10|2450|         2|
| 7369| SMITH|    CLERK|    10| 800|         3|
| 7788| SCOTT|  ANALYST|    20|3000|         1|
| 7566| JONES|  MANAGER|    20|2975|         2|
| 7876| ADAMS|    CLERK|    20|1100|         3|
| 7698| BLAKE|  MANAGER|    30|2850|         1|
| 7499| ALLEN| SALESMAN|    30|1600|         2|
| 7844|TURNER| SALESMAN|    30|1500|         3|
| 7521|  WARD| SALESMAN|    30|1250|         4|
| 7654|MARTIN| SALESMAN|    30|1250|         4|
| 7629|  ALEX| SALESMAN|    30|1150|         5|
+-----+------+---------+------+----+----------+



In [ ]:
windowSpec = Window.partitionBy(col('deptno')).orderBy(col('sal').desc())
dense_ranking_df=emp_df.select('empno', 'ename', 'job', 'deptno', 'sal',
                      F.dense_rank().over(windowSpec).alias('dense_rank'))
dense_ranking_df.show()

+-----+------+---------+------+----+----------+
|empno| ename|      job|deptno| sal|dense_rank|
+-----+------+---------+------+----+----------+
| 7839|  KING|PRESIDENT|    10|5000|         1|
| 7782| CLARK|  MANAGER|    10|2450|         2|
| 7369| SMITH|    CLERK|    10| 800|         3|
| 7788| SCOTT|  ANALYST|    20|3000|         1|
| 7566| JONES|  MANAGER|    20|2975|         2|
| 7876| ADAMS|    CLERK|    20|1100|         3|
| 7698| BLAKE|  MANAGER|    30|2850|         1|
| 7499| ALLEN| SALESMAN|    30|1600|         2|
| 7844|TURNER| SALESMAN|    30|1500|         3|
| 7521|  WARD| SALESMAN|    30|1250|         4|
| 7654|MARTIN| SALESMAN|    30|1250|         4|
| 7629|  ALEX| SALESMAN|    30|1150|         5|
+-----+------+---------+------+----+----------+



# ROWNUMBER

In [ ]:
row_df = spark.sql("""
SELECT empno, ename, job, deptno, sal,
        ROW_NUMBER() OVER (PARTITION BY deptno ORDER BY sal DESC)
         AS row_num FROM emp
""")

row_df.show()

+-----+------+---------+------+----+-------+
|empno| ename|      job|deptno| sal|row_num|
+-----+------+---------+------+----+-------+
| 7839|  KING|PRESIDENT|    10|5000|      1|
| 7782| CLARK|  MANAGER|    10|2450|      2|
| 7369| SMITH|    CLERK|    10| 800|      3|
| 7788| SCOTT|  ANALYST|    20|3000|      1|
| 7566| JONES|  MANAGER|    20|2975|      2|
| 7876| ADAMS|    CLERK|    20|1100|      3|
| 7698| BLAKE|  MANAGER|    30|2850|      1|
| 7499| ALLEN| SALESMAN|    30|1600|      2|
| 7844|TURNER| SALESMAN|    30|1500|      3|
| 7521|  WARD| SALESMAN|    30|1250|      4|
| 7654|MARTIN| SALESMAN|    30|1250|      5|
| 7629|  ALEX| SALESMAN|    30|1150|      6|
+-----+------+---------+------+----+-------+



In [ ]:
windowSpec = Window.partitionBy(col('deptno')).orderBy(col('sal').desc())

In [ ]:
emp_df.select('empno','ename','job','deptno','sal',
              F.row_number().over(windowSpec).alias('row_num')).show()

+-----+------+---------+------+----+-------+
|empno| ename|      job|deptno| sal|row_num|
+-----+------+---------+------+----+-------+
| 7839|  KING|PRESIDENT|    10|5000|      1|
| 7782| CLARK|  MANAGER|    10|2450|      2|
| 7369| SMITH|    CLERK|    10| 800|      3|
| 7788| SCOTT|  ANALYST|    20|3000|      1|
| 7566| JONES|  MANAGER|    20|2975|      2|
| 7876| ADAMS|    CLERK|    20|1100|      3|
| 7698| BLAKE|  MANAGER|    30|2850|      1|
| 7499| ALLEN| SALESMAN|    30|1600|      2|
| 7844|TURNER| SALESMAN|    30|1500|      3|
| 7521|  WARD| SALESMAN|    30|1250|      4|
| 7654|MARTIN| SALESMAN|    30|1250|      5|
| 7629|  ALEX| SALESMAN|    30|1150|      6|
+-----+------+---------+------+----+-------+



#### Question 3
We have a table of employees that includes the following fields: id, first_name, last_name, age, sex, employee_title, department, salary, target, bonus, city, address, and manager_id. We need to find the top 3 distinct salaries for each department. The output should include:

- The department name.
- The top 3 distinct salaries for each department.
- The results should be ordered alphabetically by department and then by the highest salary to the lowest salary.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rank
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder.appName("TopSalariesByDepartment").getOrCreate()
# Sample data
data = [
    (1, 'Allen', 'Wang', 55, 'F', 'Manager', 'Management', 200000, 0, 300, 'California', '23St', 1),
    (13, 'Katty', 'Bond', 56, 'F', 'Manager', 'Management', 150000, 0, 300, 'Arizona', None, 1),
    (19, 'George', 'Joe', 50, 'M', 'Manager', 'Management', 100000, 0, 300, 'Florida', '26St', 1),
    (11, 'Richerd', 'Gear', 57, 'M', 'Manager', 'Management', 250000, 0, 300, 'Alabama', None, 1),
    (10, 'Jennifer', 'Dion', 34, 'F', 'Sales', 'Sales', 100000, 200, 150, 'Alabama', None, 13),
    (18, 'Laila', 'Mark', 26, 'F', 'Sales', 'Sales', 100000, 200, 150, 'Florida', '23St', 11),
    (20, 'Sarrah', 'Bicky', 31, 'F', 'Senior Sales', 'Sales', 200000, 200, 150, 'Florida', '53St', 19),
    (21, 'Suzan', 'Lee', 34, 'F', 'Sales', 'Sales', 130000, 200, 150, 'Florida', '56St', 19),
    (22, 'Mandy', 'John', 31, 'F', 'Sales', 'Sales', 130000, 200, 150, 'Florida', '45St', 19),
    (17, 'Mick', 'Berry', 44, 'M', 'Senior Sales', 'Sales', 220000, 200, 150, 'Florida', None, 11),
    (12, 'Shandler', 'Bing', 23, 'M', 'Auditor', 'Audit', 110000, 200, 150, 'Arizona', None, 11),
    (14, 'Jason', 'Tom', 23, 'M', 'Auditor', 'Audit', 100000, 200, 150, 'Arizona', None, 11),
    (16, 'Celine', 'Anston', 27, 'F', 'Auditor', 'Audit', 100000, 200, 150, 'Colorado', None, 11),
    (15, 'Michale', 'Jackson', 44, 'F', 'Auditor', 'Audit', 70000, 150, 150, 'Colorado', None, 11),
    (6, 'Molly', 'Sam', 28, 'F', 'Sales', 'Sales', 140000, 100, 150, 'Arizona', '24St', 13),
    (7, 'Nicky', 'Bat', 33, 'F', 'Sales', 'Sales', None, None, None, None, None, None)
]
# Define columns for the employees DataFrame
columns = [
  "id",
  "first_name",
  "last_name",
  "age",
  "sex",
  "employee_title",
  "department",
  "salary",
  "target",
  "bonus", "city",
  "address",
  "manager_id"
]
# Create DataFrame
df = spark.createDataFrame(data, columns)

In [ ]:
windows_spec = Window.partitionBy("department").orderBy(col('salary').desc())

final_df = df.withColumn('rank',rank().over(windows_spec))


final_df.select('department','salary','rank').filter(col('rank')<3).show()

+----------+------+----+
|department|salary|rank|
+----------+------+----+
|     Audit|110000|   1|
|     Audit|100000|   2|
|     Audit|100000|   2|
|Management|250000|   1|
|Management|200000|   2|
|     Sales|220000|   1|
|     Sales|200000|   2|
+----------+------+----+



In [ ]:
final_df.show()

+---+----------+---------+---+---+--------------+----------+------+------+-----+----------+-------+----------+----+
| id|first_name|last_name|age|sex|employee_title|department|salary|target|bonus|      city|address|manager_id|rank|
+---+----------+---------+---+---+--------------+----------+------+------+-----+----------+-------+----------+----+
| 12|  Shandler|     Bing| 23|  M|       Auditor|     Audit|110000|   200|  150|   Arizona|   NULL|        11|   1|
| 14|     Jason|      Tom| 23|  M|       Auditor|     Audit|100000|   200|  150|   Arizona|   NULL|        11|   2|
| 16|    Celine|   Anston| 27|  F|       Auditor|     Audit|100000|   200|  150|  Colorado|   NULL|        11|   2|
| 15|   Michale|  Jackson| 44|  F|       Auditor|     Audit| 70000|   150|  150|  Colorado|   NULL|        11|   4|
| 11|   Richerd|     Gear| 57|  M|       Manager|Management|250000|     0|  300|   Alabama|   NULL|         1|   1|
|  1|     Allen|     Wang| 55|  F|       Manager|Management|200000|     

#### Question 4
Given two datasets: one containing signup details (including start and stop times) and another containing transaction details (such as amounts), determine the most profitable location based on signup duration and transaction amount

In [ ]:
signups_data = [
    (1, '2020-01-01 10:00:00', '2020-01-01 12:00:00', 101, 'New York'),
    (2, '2020-01-02 11:00:00', '2020-01-02 13:00:00', 102, 'Los Angeles'),
    (3, '2020-01-03 10:00:00', '2020-01-03 14:00:00', 103, 'Chicago'),
    (4, '2020-01-04 09:00:00', '2020-01-04 10:30:00', 101, 'San Francisco'),
    (5, '2020-01-05 08:00:00', '2020-01-05 11:00:00', 102, 'New York')
]
transactions_data = [
    (1, 1, '2020-01-01 10:30:00', 50.00),
    (2, 1, '2020-01-01 11:00:00', 30.00),
    (3, 2, '2020-01-02 11:30:00', 100.00),
    (4, 2, '2020-01-02 12:00:00', 75.00),
    (5, 3, '2020-01-03 10:30:00', 120.00),
    (6, 4, '2020-01-04 09:15:00', 80.00),
    (7, 5, '2020-01-05 08:30:00', 90.00)
]
# Define columns for signups DataFrame
signups_columns = [
  "signup_id",
  "signup_start_date",
  "signup_stop_date",
  "plan_id",
  "location"
]
signups_df = spark.createDataFrame(signups_data, signups_columns)

# Define columns for transactions DataFrame
transactions_columns = [
  "transaction_id",
  "signup_id",
  "transaction_start_date",
  "amt"
]
transactions_df = spark.createDataFrame(transactions_data, transactions_columns)


In [ ]:
signups_df.show()

+---------+-------------------+-------------------+-------+-------------+
|signup_id|  signup_start_date|   signup_stop_date|plan_id|     location|
+---------+-------------------+-------------------+-------+-------------+
|        1|2020-01-01 10:00:00|2020-01-01 12:00:00|    101|     New York|
|        2|2020-01-02 11:00:00|2020-01-02 13:00:00|    102|  Los Angeles|
|        3|2020-01-03 10:00:00|2020-01-03 14:00:00|    103|      Chicago|
|        4|2020-01-04 09:00:00|2020-01-04 10:30:00|    101|San Francisco|
|        5|2020-01-05 08:00:00|2020-01-05 11:00:00|    102|     New York|
+---------+-------------------+-------------------+-------+-------------+



In [ ]:
transactions_df.show()

+--------------+---------+----------------------+-----+
|transaction_id|signup_id|transaction_start_date|  amt|
+--------------+---------+----------------------+-----+
|             1|        1|   2020-01-01 10:30:00| 50.0|
|             2|        1|   2020-01-01 11:00:00| 30.0|
|             3|        2|   2020-01-02 11:30:00|100.0|
|             4|        2|   2020-01-02 12:00:00| 75.0|
|             5|        3|   2020-01-03 10:30:00|120.0|
|             6|        4|   2020-01-04 09:15:00| 80.0|
|             7|        5|   2020-01-05 08:30:00| 90.0|
+--------------+---------+----------------------+-----+



In [ ]:
final_df = transactions_df.join(signups_df, on="signup_id", how="inner")

In [ ]:
final_df.show()

+---------+--------------+----------------------+-----+-------------------+-------------------+-------+-------------+-----------------------+
|signup_id|transaction_id|transaction_start_date|  amt|  signup_start_date|   signup_stop_date|plan_id|     location|signup_duration_minutes|
+---------+--------------+----------------------+-----+-------------------+-------------------+-------+-------------+-----------------------+
|        1|             1|   2020-01-01 10:30:00| 50.0|2020-01-01 10:00:00|2020-01-01 12:00:00|    101|     New York|                  120.0|
|        1|             2|   2020-01-01 11:00:00| 30.0|2020-01-01 10:00:00|2020-01-01 12:00:00|    101|     New York|                  120.0|
|        2|             3|   2020-01-02 11:30:00|100.0|2020-01-02 11:00:00|2020-01-02 13:00:00|    102|  Los Angeles|                  120.0|
|        2|             4|   2020-01-02 12:00:00| 75.0|2020-01-02 11:00:00|2020-01-02 13:00:00|    102|  Los Angeles|                  120.0|
|     

In [ ]:
from pyspark.sql.functions import col, unix_timestamp, when, avg
signups_df = signups_df.withColumn(
    "signup_duration_minutes",
    (unix_timestamp("signup_stop_date") - unix_timestamp("signup_start_date")) / 60
)

In [ ]:
transaction_avg_df = transactions_df.groupBy("signup_id").agg(avg("amt").alias("avg_transaction_amount"))

In [ ]:
# Join the signups with transaction averages
joined_df = signups_df.join(transaction_avg_df, on="signup_id", how="inner")

In [ ]:
result_df = joined_df.groupBy("location").agg(
    avg("signup_duration_minutes").alias("avg_duration"),
    avg("avg_transaction_amount").alias("avg_transaction_amount")
)

In [ ]:
result_df.show()

+-------------+------------+----------------------+
|     location|avg_duration|avg_transaction_amount|
+-------------+------------+----------------------+
|  Los Angeles|       120.0|                  87.5|
|San Francisco|        90.0|                  80.0|
|      Chicago|       240.0|                 120.0|
|     New York|       150.0|                  65.0|
+-------------+------------+----------------------+



In [ ]:
result_df = result_df.withColumn(
    "ratio",
    when(col("avg_duration") != 0, col("avg_transaction_amount") / col("avg_duration")).otherwise(0)
)

In [ ]:
result_df = result_df.orderBy(col("ratio"), ascending=False)

# Show the final result
result_df.show(truncate=False)

+-------------+------------+----------------------+-------------------+
|location     |avg_duration|avg_transaction_amount|ratio              |
+-------------+------------+----------------------+-------------------+
|San Francisco|90.0        |80.0                  |0.8888888888888888 |
|Los Angeles  |120.0       |87.5                  |0.7291666666666666 |
|Chicago      |240.0       |120.0                 |0.5                |
|New York     |150.0       |65.0                  |0.43333333333333335|
+-------------+------------+----------------------+-------------------+



#### Question 5
The problem is to calculate the minimum number of platforms required at a train station based on the given arrival_times and departure_times.

In [ ]:
arrivals_data = [
    (1, '2024-11-17 08:00'),
    (2, '2024-11-17 08:05'),
    (3, '2024-11-17 08:05'),
    (4, '2024-11-17 08:10'),
    (5, '2024-11-17 08:10'),
    (6, '2024-11-17 12:15'),
    (7, '2024-11-17 12:20'),
    (8, '2024-11-17 12:25'),
    (9, '2024-11-17 15:00'),
    (10, '2024-11-17 15:00'),
    (11, '2024-11-17 15:00'),
    (12, '2024-11-17 15:06'),
    (13, '2024-11-17 20:00'),
    (14, '2024-11-17 20:10')
]
departures_data = [
    (1, '2024-11-17 08:15'),
    (2, '2024-11-17 08:10'),
    (3, '2024-11-17 08:20'),
    (4, '2024-11-17 08:25'),
    (5, '2024-11-17 08:20'),
    (6, '2024-11-17 13:00'),
    (7, '2024-11-17 12:25'),
    (8, '2024-11-17 12:30'),
    (9, '2024-11-17 15:05'),
    (10, '2024-11-17 15:10'),
    (11, '2024-11-17 15:15'),
    (12, '2024-11-17 15:15'),
    (13, '2024-11-17 20:15'),
    (14, '2024-11-17 20:15')
]

# Define schema for the data
arrival_columns = ['train_id', 'arrival_time']
departure_columns = ['train_id', 'departure_time']
# Create DataFrames
arrivals_df = spark.createDataFrame(arrivals_data, arrival_columns)
departures_df = spark.createDataFrame(departures_data, departure_columns)

In [ ]:
arrivals_df = arrivals_df.withColumn('arrival_time', col('arrival_time').cast('timestamp'))
departures_df = departures_df.withColumn('departure_time', col('departure_time').cast('timestamp'))

In [ ]:
arrivals_df.show()

+--------+-------------------+
|train_id|       arrival_time|
+--------+-------------------+
|       1|2024-11-17 08:00:00|
|       2|2024-11-17 08:05:00|
|       3|2024-11-17 08:05:00|
|       4|2024-11-17 08:10:00|
|       5|2024-11-17 08:10:00|
|       6|2024-11-17 12:15:00|
|       7|2024-11-17 12:20:00|
|       8|2024-11-17 12:25:00|
|       9|2024-11-17 15:00:00|
|      10|2024-11-17 15:00:00|
|      11|2024-11-17 15:00:00|
|      12|2024-11-17 15:06:00|
|      13|2024-11-17 20:00:00|
|      14|2024-11-17 20:10:00|
+--------+-------------------+



In [ ]:
arrivals_df = arrivals_df.withColumn('event_type', F.lit(1))
departures_df = departures_df.withColumn('event_type', F.lit(-1))

In [ ]:
# Union both DataFrames into one, marking events as either arrival or departure
all_events_df = arrivals_df.select('train_id', 'arrival_time', 'event_type') \
    .withColumnRenamed('arrival_time', 'event_time') \
    .union(departures_df.select('train_id', 'departure_time', 'event_type') \
    .withColumnRenamed('departure_time', 'event_time'))
all_events_df = all_events_df.orderBy('event_time', F.col('event_type').desc())

# Use a window function to calculate the running total of platforms needed at each time
window_spec = Window.orderBy('event_time', F.col('event_type').desc())  # Same order as the events

# Calculate running sum of platforms needed
all_events_df = all_events_df.withColumn('platforms_needed', F.sum('event_type').over(window_spec))

# Find the maximum platforms_needed at any given time
max_platforms = all_events_df.agg(F.max('platforms_needed')).collect()[0][0]

# Output the result
print(f"The minimum number of platforms required: {max_platforms}")


The minimum number of platforms required: 5


#### Question 6
IBM is working on a new feature to analyze user purchasing behavior for all Fridays in the first quarter of the year. For each Friday separately, calculate the average amount users have spent per order. The output should contain the week number of that Friday and average amount spent.

###### PySpark Approach:
- Filter Fridays in Q1: First, we need to identify the Fridays (where day_name == 'Friday') and restrict the data to the first quarter of the year (dates from January to March).
- Calculate Week Number: We will use the weekofyear() function in PySpark to extract the week number from the date.
- Group and Aggregate: We group the data by week number and compute the average of amount_spent for each week.
PySpark Code Implementation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, weekofyear

# Initialize Spark session
spark = SparkSession.builder.appName("UserPurchasesQ1Fridays").getOrCreate()
# Sample data for user purchases
user_purchases_data = [
    (1047, '2023-01-01', 288, 'Sunday'),
    (1099, '2023-01-04', 803, 'Wednesday'),
    (1055, '2023-01-07', 546, 'Saturday'),
    (1040, '2023-01-10', 680, 'Tuesday'),
    (1052, '2023-01-13', 889, 'Friday'),
    (1052, '2023-01-13', 596, 'Friday'),
    (1016, '2023-01-16', 960, 'Monday'),
    (1023, '2023-01-17', 861, 'Tuesday'),
    (1010, '2023-01-19', 758, 'Thursday'),
    (1013, '2023-01-19', 346, 'Thursday'),
    (1069, '2023-01-21', 541, 'Saturday'),
    (1030, '2023-01-22', 175, 'Sunday'),
    (1034, '2023-01-23', 707, 'Monday'),
    (1019, '2023-01-25', 253, 'Wednesday'),
    (1052, '2023-01-25', 868, 'Wednesday'),
    (1095, '2023-01-27', 424, 'Friday'),
    (1017, '2023-01-28', 755, 'Saturday'),
    (1010, '2023-01-29', 615, 'Sunday'),
    (1063, '2023-01-31', 534, 'Tuesday'),
    (1019, '2023-02-03', 185, 'Friday'),
    (1019, '2023-02-03', 995, 'Friday'),
    (1092, '2023-02-06', 796, 'Monday'),
    (1058, '2023-02-09', 384, 'Thursday'),
    (1055, '2023-02-12', 319, 'Sunday'),
    (1090, '2023-02-15', 168, 'Wednesday'),
    (1090, '2023-02-18', 146, 'Saturday'),
    (1062, '2023-02-21', 193, 'Tuesday'),
    (1023, '2023-02-24', 259, 'Friday')
]
# Define schema for user purchases
columns_user_purchases = ["user_id", "date", "amount_spent", "day_name"]
# Create DataFrame for user purchases
df_user_purchases = spark.createDataFrame(user_purchases_data, columns_user_purchases)


In [ ]:
friday_data = df_user_purchases.filter((col('day_name')=='Friday') & (col('date')>='2023-01-01') & (col('date')<='2023-03-31'))

In [ ]:
friday_data = friday_data.withColumn('weekNum', weekofyear(col('date')))

In [ ]:
friday_data.show()

+-------+----------+------------+--------+-------+
|user_id|      date|amount_spent|day_name|weekNum|
+-------+----------+------------+--------+-------+
|   1052|2023-01-13|         889|  Friday|      2|
|   1052|2023-01-13|         596|  Friday|      2|
|   1095|2023-01-27|         424|  Friday|      4|
|   1019|2023-02-03|         185|  Friday|      5|
|   1019|2023-02-03|         995|  Friday|      5|
|   1023|2023-02-24|         259|  Friday|      8|
+-------+----------+------------+--------+-------+



In [ ]:
friday_data.groupBy('weekNum').agg({"amount_spent": "avg"}).withColumnRenamed("avg(amount_spent)", "avg_amount_spent").show()

+-------+----------------+
|weekNum|avg_amount_spent|
+-------+----------------+
|      2|           742.5|
|      5|           590.0|
|      4|           424.0|
|      8|           259.0|
+-------+----------------+



#### Question 7
Find the number of Apple product users (MacBook-Pro, iPhone 5s, iPad-air) and the total number of users with any device, grouped by language. Output the language along with the total number of Apple users and users with any device. Order the results by the number of total users in descending order.

###### Solution
**Apple Users CTE:**

- We filter the playbook_events table for users who have one of the Apple devices (MacBook-Pro, iPhone 5s, iPad-air).
- We then join this filtered data with the playbook_users table to get the user details and group by language.
- We count the distinct user_id per language to get the number of Apple product users for each language.
**Total Users CTE:**

We join playbook_events with playbook_users on user_id to get all the users' details across any device.
We group by language and count distinct user_id to get the total number of users per language.
Final Query:

We perform a left join between apple_users_df and total_users_df on language to merge the counts.
We use coalesce to handle cases where there are no Apple users for a language, setting the count to 0 in such cases.
The results are ordered by the number of total users in descending order.
PySpark Code Implementation

In [ ]:
users_data = [
    (1, '2024-01-01 08:00:00', 101, 'English', '2024-01-05 10:00:00', 'Active'),
    (2, '2024-01-02 09:00:00', 102, 'Spanish', '2024-01-06 11:00:00', 'Inactive'),
    (3, '2024-01-03 10:00:00', 103, 'French', '2024-01-07 12:00:00', 'Active'),
    (4, '2024-01-04 11:00:00', 104, 'English', '2024-01-08 13:00:00', 'Active'),
    (5, '2024-01-05 12:00:00', 105, 'Spanish', '2024-01-09 14:00:00', 'Inactive')
]
# Sample data for playbook_events
events_data = [
    (1, '2024-01-05 14:00:00', 'Click', 'Login', 'USA', 'MacBook-Pro'),
    (2, '2024-01-06 15:00:00', 'View', 'Dashboard', 'Spain', 'iPhone 5s'),
    (3, '2024-01-07 16:00:00', 'Click', 'Logout', 'France', 'iPad-air'),
    (4, '2024-01-08 17:00:00', 'Purchase', 'Subscription', 'USA', 'Windows-Laptop'),
    (5, '2024-01-09 18:00:00', 'Click', 'Login', 'Spain', 'Android-Phone')
]
# Define schema for users and events data
users_columns = [
  "user_id",
  "created_at",
  "company_id",
  "language",
  "activated_at",
  "state"
]
events_columns = [
  "user_id",
  "occurred_at",
  "event_type",
  "event_name",
  "location",
  "device"
]
# Create DataFrames
users_df = spark.createDataFrame(users_data, users_columns)
events_df = spark.createDataFrame(events_data, events_columns)

In [ ]:
# Define the Apple devices
apple_devices = ['MacBook-Pro', 'iPhone 5s', 'iPad-air']

# Apple Users CTE: Filter users with Apple devices and count distinct users per language
apple_users_df = (events_df.filter(col('device').isin(apple_devices))
                  .join(users_df, 'user_id', 'inner')
                  .groupBy('language')
                  .agg(countDistinct('user_id').alias('apple_users')))

In [ ]:
apple_users_df.show()

+--------+-----------+
|language|apple_users|
+--------+-----------+
| English|          1|
| Spanish|          1|
|  French|          1|
+--------+-----------+



In [ ]:
# Total Users CTE: Count distinct users per language for any device
total_users_df = (events_df.join(users_df, 'user_id', 'inner')
                  .groupBy('language')
                  .agg(countDistinct('user_id').alias('total_users')))

In [ ]:
total_users_df.show()

+--------+-----------+
|language|total_users|
+--------+-----------+
| English|          2|
| Spanish|          2|
|  French|          1|
+--------+-----------+



In [ ]:
# Join Apple users and Total users, and handle nulls with COALESCE
final_df = (apple_users_df.join(total_users_df, 'language', 'outer')
            .select('language',
                    coalesce('apple_users', lit(0)).alias('apple_users'),
                    coalesce('total_users', lit(0)).alias('total_users'))
            .orderBy(col('total_users').desc()))

# Show the result
final_df.show()

+--------+-----------+-----------+
|language|apple_users|total_users|
+--------+-----------+-----------+
| English|          1|          2|
| Spanish|          1|          2|
|  French|          1|          1|
+--------+-----------+-----------+



#### Question 8
We have two datasets:

- Sessions Table: Contains records of when users started their sessions.
- Order Summary Table: Contains records of orders placed by users along with their values.
We want to:

Find users who started a session and placed an order on the same day.
Calculate the total number of orders and the total order value for those users.
#### Solution
We will convert the session and order dates to just the date (ignoring time) using the to_date() function. This step ensures that we can match sessions and orders that happened on the same calendar day, which is the core of our analysis.

Finally, we’ll join the two DataFrames on user_id and matching dates. After grouping the data by user_id and session date, we’ll calculate the total number of orders and total order value for each user on the same day. We’ll filter out users who didn’t place any orders, and then display the results.

In [ ]:
# Sample data for sessions
sessions_data = [
    (1, 1, '2024-01-01 00:00:00'),
    (2, 2, '2024-01-02 00:00:00'),
    (3, 3, '2024-01-05 00:00:00'),
    (4, 3, '2024-01-05 00:00:00'),
    (5, 4, '2024-01-03 00:00:00'),
    (6, 4, '2024-01-03 00:00:00'),
    (7, 5, '2024-01-04 00:00:00'),
    (8, 5, '2024-01-04 00:00:00'),
    (9, 3, '2024-01-05 00:00:00'),
    (10, 5, '2024-01-04 00:00:00')
]
# Sample data for orders
orders_data = [
    (1, 1, 152, '2024-01-01 00:00:00'),
    (2, 2, 485, '2024-01-02 00:00:00'),
    (3, 3, 398, '2024-01-05 00:00:00'),
    (4, 3, 320, '2024-01-05 00:00:00'),
    (5, 4, 156, '2024-01-03 00:00:00'),
    (6, 4, 121, '2024-01-03 00:00:00'),
    (7, 5, 238, '2024-01-04 00:00:00'),
    (8, 5, 70, '2024-01-04 00:00:00'),
    (9, 3, 152, '2024-01-05 00:00:00'),
    (10, 5, 171, '2024-01-04 00:00:00')
]
session_columns = ["session_id", "user_id", "session_date"]
orders_columns = ["order_id", "user_id", "order_value", "order_date"]
# Convert data into DataFrames
sessions_df = spark.createDataFrame(sessions_data, session_columns)
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [ ]:
sessions_df = sessions_df.withColumn('session_date', to_date(col('session_date')))

orders_df = orders_df.withColumn('order_date', to_date(col('order_date')))

In [ ]:
joined_df = sessions_df.join(orders_df, on=['user_id'], how='left')

In [ ]:
joined_df = joined_df.withColumn('isSameDayOrder', when(col('session_date') == col('order_date'),True))

In [ ]:
joined_df.filter(col('isSameDayOrder')==True).groupBy('user_id').agg({'order_value': 'sum', 'order_id': 'count'}).withColumnRenamed('sum(order_value)', 'total_order_value').withColumnRenamed('count(order_id)', 'total_orders').show()

+-------+------------+-----------------+
|user_id|total_orders|total_order_value|
+-------+------------+-----------------+
|      1|           1|              152|
|      3|           9|             2610|
|      2|           1|              485|
|      4|           4|              554|
|      5|           9|             1437|
+-------+------------+-----------------+



In [ ]:
sessions_df.show()

+----------+-------+------------+
|session_id|user_id|session_date|
+----------+-------+------------+
|         1|      1|  2024-01-01|
|         2|      2|  2024-01-02|
|         3|      3|  2024-01-05|
|         4|      3|  2024-01-05|
|         5|      4|  2024-01-03|
|         6|      4|  2024-01-03|
|         7|      5|  2024-01-04|
|         8|      5|  2024-01-04|
|         9|      3|  2024-01-05|
|        10|      5|  2024-01-04|
+----------+-------+------------+



In [ ]:
orders_df.show()

+--------+-------+-----------+----------+
|order_id|user_id|order_value|order_date|
+--------+-------+-----------+----------+
|       1|      1|        152|2024-01-01|
|       2|      2|        485|2024-01-02|
|       3|      3|        398|2024-01-05|
|       4|      3|        320|2024-01-05|
|       5|      4|        156|2024-01-03|
|       6|      4|        121|2024-01-03|
|       7|      5|        238|2024-01-04|
|       8|      5|         70|2024-01-04|
|       9|      3|        152|2024-01-05|
|      10|      5|        171|2024-01-04|
+--------+-------+-----------+----------+

